# ДЗ 2. Агент с графом знаний

Агент-планировщик из ДЗ 1 получает базу знаний турагентства «Попутный ветер»: отели, гиды, экскурсии, туры и правила агентства. Многие вопросы к ней не решаются одним поиском по тексту: чтобы узнать, на каких языках можно пройти экскурсию, нужно найти её гида, а языки гида записаны в другом документе. Для таких вопросов вы построите граф знаний: модель извлечёт из документов сущности и связи, вы соберёте из них граф, сделаете по нему инструмент поиска и дадите агенту выбирать между поиском по тексту и по графу. В конце сравните агента с обычным RAG и разберёте ошибки.

Как устроено задание:

1. **Модель читает документы.** По каждому документу модель возвращает список сущностей (гид, отель, экскурсия…) и связей между ними. Вы пишете промпт и проверяете её ответы. Графа пока нет: есть 47 отдельных списков.
2. **Собираем граф.** Из этих списков вы строите один граф: одна и та же сущность из разных документов становится одной вершиной, и связи из разных документов складываются в цепочки.
3. **Агент ищет по графу и по тексту.** По графу вы делаете инструмент поиска и даёте агенту выбирать между ним и поиском по тексту.
4. **Сравнение.** Обычный RAG, агент только с текстом и агент с графом на одних и тех же вопросах.
5. **Разбор ошибок** — на каком из этих шагов они возникли.

Все факты в базе вымышлены: модель не может знать их заранее.

### Что сдать

Только этот ноутбук с выполненными ячейками: все выводы должны быть видны.

### Правила

* Код пишите в местах с пометкой `# TODO`. **Не меняйте имена и сигнатуры** функций и классов: по ним работают самопроверки.
* После большинства заданий стоит ячейка `check(...)`. Она проверяет код без обращения к LLM и печатает ✅ или ❌ с причиной. ✅ нужен для полного балла, но его мало: проверяющий смотрит ещё выводы и тексты.
* Тексты пишите своими словами по выводам своих ячеек: номера вопросов, цифры, вызовы агента.
* Библиотеки: `langchain`, `langchain-openai`, `networkx`, `rank_bm25`.

## 0. Подготовка

In [20]:
%pip install -q "langchain>=1.4,<2" "langchain-openai>=1.7,<2" networkx rank_bm25

In [34]:
import json
import os
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from getpass import getpass
from pathlib import Path
from typing import Any, Callable, Optional

import networkx as nx
import numpy as np
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import BaseTool, ToolException
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langgraph.errors import GraphRecursionError
from pydantic import BaseModel, Field
from rank_bm25 import BM25Okapi

try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["LLM_BASE_URL"] = "https://api.openai.com/v1" # "https://api.groq.com/openai/v1"
    os.environ["LLM_MODEL"] = "gpt-6.2-astra" # "openai/gpt-oss-120b"
except Exception as e:
    print(e)

missing = [name for name in ["hw02_checks.py", "hw02_data.json"] if not Path(name).exists()]
if missing:
    try:
        from google.colab import files

        print("Загрузите файлы:", ", ".join(missing))
        files.upload()
    except ImportError as error:
        raise FileNotFoundError(f"Положите рядом с ноутбуком: {', '.join(missing)}") from error

from hw02_checks import ScriptedChatModel, check

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:11434/v1")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-oss:120b-cloud")
API_KEY = os.environ.setdefault("LLM_API_KEY", "ollama")

llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL, api_key=os.environ["LLM_API_KEY"], temperature=1,
                 timeout=120, max_retries=2)
print(llm.invoke("Ответь одним словом: готово").content)

готово


### Данные

В `hw02_data.json`:

* `DOCS` — 47 документов: карточки отелей, гидов, экскурсий, туров и правила агентства. Поля `doc_id`, `title`, `kind`, `text`.
* `SCHEMA` — схема графа: 6 типов сущностей и 7 типов связей с указанием, между какими типами связь допустима.
* `EVAL` — 24 вопроса. У каждого `hops` — сколько шагов по связям нужно для ответа (0 — ответа в базе нет), `gold_doc_ids` и `key_facts` — что должно быть в ответе.

Ниже — готовые вспомогательные функции, их менять не нужно. Среди них `parallel_map`: граф строится по одному запросу к модели на каждый документ (47 запросов), а сравнение систем в части 4 — это ещё около 80 запросов. `parallel_map` отправляет их по 8 одновременно, иначе эти ячейки шли бы по полчаса.

In [35]:
DATA = json.loads(Path("hw02_data.json").read_text(encoding="utf-8"))
DOCS: list[dict] = DATA["documents"]
SCHEMA: dict = DATA["schema"]
EVAL: list[dict] = DATA["eval"]
DOC_BY_ID = {doc["doc_id"]: doc for doc in DOCS}

MAX_WORKERS = 4  # сколько запросов к модели идёт одновременно; при ошибках 429 от прокси уменьшите


def parallel_map(fn: Callable, items: list, desc: str = "") -> list:
    """fn(item) для всех items в несколько потоков; результаты в исходном порядке."""
    results, errors, done = [None] * len(items), [], 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(fn, item): index for index, item in enumerate(items)}
        for future in as_completed(futures):
            index = futures[future]
            try:
                results[index] = future.result()
            except Exception as error:
                errors.append((index, error))
            done += 1
            if done % max(1, len(items) // 5) == 0 or done == len(items):
                print(f"{desc}: готово {done}/{len(items)}", flush=True)
    if errors:
        index, error = min(errors, key=lambda pair: pair[0])
        raise RuntimeError(f"{len(errors)} из {len(items)} задач упали; первая — элемент {index}: {error!r}") from error
    return results


def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zа-яё0-9]+", text.lower())


def extract_citations(answer: str) -> list[str]:
    """doc_id из квадратных скобок в ответе, уникальные, по порядку."""
    found: list[str] = []
    for group in re.findall(r"\[([^\]]+)\]", answer or ""):
        for token in re.split(r"[,;\s]+", group):
            if re.fullmatch(r"[a-z]+_[a-z0-9_]+", token) and token not in found:
                found.append(token)
    return found


def has_no_info(answer: str) -> bool:
    return bool(re.search(r"нет информаци|нет данных|не нашёл|не нашел|не найден|нет в базе|не располага|отсутству|такого .* нет",
                          answer or "", re.IGNORECASE))


print("Типы сущностей:", ", ".join(SCHEMA["entity_types"]))
for name, rule in SCHEMA["relations"].items():
    print(f"  {name}: {rule['source']} → {rule['target']} — {rule['meaning']}")
print("\nДокументов:", len(DOCS), "| вопросов:", len(EVAL))
print("\nПример:", DOC_BY_ID["e_kzn_food"]["text"])

Типы сущностей: Город, Отель, Экскурсия, Гид, Язык, Тур
  находится_в: Отель|Экскурсия → Город — отель стоит в городе / экскурсия проходит в городе
  работает_в: Гид → Город — город, где работает гид
  ведёт: Гид → Экскурсия — гид проводит экскурсию
  говорит_на: Гид → Язык — гид может вести экскурсии на этом языке
  стартует_от: Экскурсия → Отель — сбор группы у отеля или в его холле
  включает: Тур → Экскурсия — экскурсия входит в программу тура
  проживание_в: Тур → Отель — участники тура живут в этом отеле

Документов: 47 | вопросов: 24

Пример: «Гастротур по Казани» — авторская экскурсия в Казани: чак-чак, эчпочмак и губадия. Длительность 3 ч, стоимость 2900 рублей с человека.

Встреча у входа на Колхозный рынок. Ведёт экскурсию Алия.


## 1. Чтение документов моделью (1.75 балла)

В этой части граф ещё не строится. Модель читает каждый документ отдельно и перечисляет, какие сущности в нём упомянуты и как они связаны, по схеме `SCHEMA`. Для этого нужно:

1. написать промпт;
2. проверить правильность данных, извлечённых моделью.

На выходе должно получиться 47 проверенных списков сущностей и связей, по одному на документ. В части 2 вы соберёте из них граф.

### 1.1. Промпт извлечения (0.5 балла)

Граф строится из текста: модель читает документ и перечисляет, какие сущности в нём упомянуты и как они связаны.

Например, из карточки экскурсии `e_kzn_food` («Гастротур по Казани» — авторская экскурсия в Казани … Ведёт экскурсию Алия.) должно получиться примерно такое:

```json
{"entities": [{"name": "Гастротур по Казани", "type": "Экскурсия"},
              {"name": "Казань", "type": "Город"},
              {"name": "Алия", "type": "Гид"}],
 "relations": [{"source": "Гастротур по Казани", "relation": "находится_в", "target": "Казань",
                "evidence": "авторская экскурсия в Казани"},
               {"source": "Алия", "relation": "ведёт", "target": "Гастротур по Казани",
                "evidence": "Ведёт экскурсию Алия"}]}
```

От этого промпта зависит качество всего графа: то, что модель здесь пропустит, не попадёт в граф, а то, что придумает, окажется в нём лишним.

Напишите `build_extraction_prompt(doc, schema) -> list[dict]` — сообщения для `llm.invoke` в формате `[{"role": "system", "content": ...}, {"role": "user", "content": ...}]`. В промпте должны быть:

* все типы сущностей и связей схемы с пояснениями, для связей — между какими типами она допустима;
* правила записи имён: как писать город (именительный падеж), язык, нужно ли убирать кавычки и слова «отель», «гостиница»;
* просьба извлекать только то, что сказано в документе;
* ответ — только JSON вида `{"entities": [{"name", "type"}], "relations": [{"source", "relation", "target", "evidence"}]}`, где `evidence` — точная цитата из документа, подтверждающая связь;
* текст документа целиком.

In [36]:
SYSTEM_PROMPT = """
Извлеки только сущности и связи, явно указанные в тексте.
Схема с типами, пояснениями и направлением связей (source → target, | означает один из типов):
{{SCHEMA}}.
Названия городов и языков пиши в именительном падеже: Казань, Санкт-Петербург, английский, татарский.
Языки пиши со строчной буквы, без слова «язык».
Убирай внешние кавычки и родовые слова «отель», «гостиница», «гид», «экскурсия», «тур» перед именем.
Сохраняй полное имя, если оно дано в тексте.
Каждый конец связи должен совпадать с именем в entities.
Возвращай только JSON без Markdown, в формате
{
    "entities": [{"name": "...", "type": "..."}],
    "relations": [{"source": "...", "relation": "...", "target": "...", "evidence": "..."}]
}
evidence — точная непрерывная цитата из текста, подтверждающая связь.
Если несколько связей подтверждаются одним перечислением, скопируй предложение целиком и используй его без изменений в evidence каждой связи.
Значения source и target копируй посимвольно из поля name соответствующей сущности в entities.
"""

def build_extraction_prompt(doc: dict, schema: dict) -> list[dict]:
    # TODO 1.1
    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT.replace("{{SCHEMA}}", json.dumps(schema, ensure_ascii=False))
        },
        {
            "role": "user",
            "content": doc["text"]
        }
    ]


check("1.1", build_extraction_prompt=build_extraction_prompt)

✅ Задание 1.1: базовые тесты пройдены


### 1.2. Разбор и проверка ответа (0.75 балла)

Ответ модели на промпт из 1.1 — это текст, внутри которого, если всё прошло хорошо, лежит JSON со списками сущностей и связей. Чтобы собрать из него граф, этот JSON нужно достать из текста и проверить. Всё, что пройдёт проверку, попадёт в итоговый граф.

Модель может ошибаться: оборачивает JSON в текст, придумывает типы, путает направление связей, приводит «цитаты», которых нет в документе.

Напишите `parse_extraction(text, doc_text, schema) -> dict | None`, который:

* находит JSON в ответе (в том числе внутри текста или блока кода); если JSON нет или в нём нет списков `entities` и `relations` — `None`; функция не падает ни на каком ответе;
* оставляет сущности только с типами из схемы;
* оставляет связь, только если: связь есть в схеме; оба конца есть среди сущностей этого ответа; типы концов допустимы для этой связи (`"Отель|Экскурсия"` — любой из двух); `evidence` действительно встречается в тексте документа (без учёта регистра, лишних пробелов и разницы ё/е);
* возвращает `{"entities": [...], "relations": [...], "dropped": [...]}`, где `dropped` — отброшенные связи с полем `reason`: почему отброшена. Это понадобится в 1.3.

In [37]:
def parse_extraction(text: str, doc_text: str, schema: dict) -> Optional[dict]:
    # TODO 1.2
    decoder = json.JSONDecoder()
    for match in re.finditer(r"\{", text):
        try:
            data, _ = decoder.raw_decode(text, match.start())
        except (ValueError, RecursionError):
            continue
        if isinstance(data, dict) and isinstance(data.get("entities"), list) and isinstance(data.get("relations"), list):
            break
    else:
        return None
    entities = [e for e in data["entities"] if isinstance(e, dict) and isinstance(e.get("name"), str) and isinstance(e.get("type"), str) and e["type"] in schema["entity_types"]]
    types = {e["name"]: e["type"] for e in entities}
    normalize = lambda value: " ".join(value.lower().replace("ё", "е").split())
    source_text = normalize(doc_text)
    relations, dropped = [], []
    for edge in data["relations"]:
        if not isinstance(edge, dict):
            dropped.append({"reason": "связь не является объектом", "value": edge})
            continue
        source, relation, target, evidence = (edge.get(key) for key in ("source", "relation", "target", "evidence"))
        if not isinstance(relation, str) or relation not in schema["relations"]:
            reason = "связь отсутствует в схеме"
        elif not isinstance(source, str) or not isinstance(target, str) or source not in types or target not in types:
            reason = "конец отсутствует среди сущностей"
        elif types[source] not in schema["relations"][relation]["source"].split("|") or types[target] not in schema["relations"][relation]["target"].split("|"):
            reason = "неподходящие типы концов"
        elif not isinstance(evidence, str) or not normalize(evidence) or normalize(evidence) not in source_text:
            reason = "цитата отсутствует в документе"
        else:
            relations.append(edge)
            continue
        dropped.append({**edge, "reason": reason})
    return {"entities": entities, "relations": relations, "dropped": dropped}


check("1.2", parse_extraction=parse_extraction)

✅ Задание 1.2: базовые тесты пройдены


Запустим модель на всех документах (47 запросов к модели, по 8 одновременно).

Результат сохраняется в `graph_extraction.json`: при перезапуске ноутбука запросы не повторяются. Чтобы извлечь заново (например, после правки промпта), поставьте `REEXTRACT = True`.

In [38]:
REEXTRACT = True
CACHE = Path("graph_extraction.json")


def extract_one(doc: dict) -> dict:
    reply = llm.invoke(build_extraction_prompt(doc, SCHEMA)).text
    parsed = parse_extraction(reply, doc["text"], SCHEMA)
    return {"doc_id": doc["doc_id"], "raw": reply, "parsed": parsed}


if CACHE.exists() and not REEXTRACT:
    EXTRACTIONS = json.loads(CACHE.read_text(encoding="utf-8"))
    print("Загружено из", CACHE)
else:
    EXTRACTIONS = parallel_map(extract_one, DOCS, "извлечение")
    CACHE.write_text(json.dumps(EXTRACTIONS, ensure_ascii=False, indent=1), encoding="utf-8")

failed = [e["doc_id"] for e in EXTRACTIONS if e["parsed"] is None]
parsed = [e for e in EXTRACTIONS if e["parsed"] is not None]
entity_types = Counter(ent["type"] for e in parsed for ent in e["parsed"]["entities"])
relation_types = Counter(rel["relation"] for e in parsed for rel in e["parsed"]["relations"])
dropped = [(e["doc_id"], d) for e in parsed for d in e["parsed"]["dropped"]]

print("Ответ не разобрался:", failed or "ни разу")
print("Сущностей по типам:", dict(entity_types))
print("Связей по типам:", dict(relation_types))
print(f"Отброшено связей: {len(dropped)}", dict(Counter(d['reason'] for _, d in dropped)))
for doc_id, d in dropped[:12]:
    print(f"  [{doc_id}] {d.get('source')} —{d.get('relation')}→ {d.get('target')} | {d['reason']} | «{str(d.get('evidence'))[:60]}»")

извлечение: готово 9/47
извлечение: готово 18/47
извлечение: готово 27/47
извлечение: готово 36/47
извлечение: готово 45/47
извлечение: готово 47/47
Ответ не разобрался: ни разу
Сущностей по типам: {'Отель': 31, 'Город': 45, 'Гид': 26, 'Язык': 25, 'Экскурсия': 26, 'Тур': 5}
Связей по типам: {'находится_в': 27, 'работает_в': 11, 'говорит_на': 25, 'стартует_от': 12, 'ведёт': 15, 'проживание_в': 7, 'включает': 11}
Отброшено связей: 0 {}


### 1.3. Что извлекла модель (0.5 балла)

По выводу выше ответьте (4–6 предложений), ссылаясь на `doc_id`:

1. Каких связей и сущностей получилось меньше или больше, чем вы ожидали, глядя на документы?
2. Какие связи отброшены и почему — это ошибки модели или ваша проверка слишком строгая? Приведите два-три примера.
3. Что вы поменяли в промпте 1.1 после первого извлечения и что от этого изменилось? Если не меняли — что поменяли бы.

**Ваш ответ:**

1. Я ожидала много упоминаний городов и отелей, так как они повторяются в документах разных типов. Больше всего упоминаний городов (45), затем отелей (31), хотя отдельных карточек отелей всего 12. Например, отель из `h_petro` также упоминается под названием «Петроградка» в `t_capitals` и `e_spb_yards`. Из связей самой частой оказалась `находится_в` (27), что совпало с моей оценкой по числу карточек отелей (12) и экскурсий (15). Связей `работает_в` получилось 11 — столько же, сколько карточек гидов.

2. Ни одна связь не была отброшена, все отработало правильно.

3. После первого извлечения я добавила такую инструкцию: "Если несколько связей подтверждаются одним перечислением, скопируй предложение целиком и используй его без изменений в evidence каждой связи. Значения source и target копируй посимвольно из поля name соответствующей сущности в entities." Она помогла точнее извлекать evidence для последующей проверки правильности связей.

## 2. Собираем граф из ответов модели (2.75 балла)

После части 1 у вас 47 отдельных списков сущностей и связей между ними, по одному списку на документ. Чтобы ответить на вопрос пользователя, их нужно соединить в один граф. Тогда от экскурсии можно перейти по ребру «ведёт» к гиду, а от гида по рёбрам «говорит_на» — к языкам, хотя эти факты записаны в разных документах.

Этим займётся класс `KnowledgeGraph`: он собирает граф в `networkx.MultiDiGraph` (`self.graph`) и ищет по нему. Обход соседей и путей даёт `networkx`, вы пишете логику предметной области. В части 3 по этому классу вы сделаете инструмент поиска для агента.

* Вершина — сущность. Атрибуты: `name` (самое полное из встреченных имён), `type`, `aliases` (множество нормализованных имён), `doc_ids` (множество документов, где она встретилась). Идентификатор вершины — на ваше усмотрение.
* Ребро — связь. Атрибуты: `relation`, `doc_id` (откуда извлечена), `evidence` (цитата).

Методы:

* **2.1 `resolve(name, etype) -> node | None` (1).** Ищет среди вершин того же типа ту, к которой относится упоминание. Зачем: одна сущность в разных документах названа по-разному, и без слияния в графе окажутся две вершины.

  > Тогда цепочка рвётся: в карточке экскурсии гид — «Алия», а языки записаны у вершины «Алия Гарипова», и от экскурсии до языков не дойти. Примеры разных написаний: «Казань» и «казань», «Кабан» и «Гостиница „Кабан“», «Алия» и «Алия Гарипова», «Петроградка» и «Отель „Петроградская сторона“».

  Правила сопоставления придумайте сами: нормализация, совпадение по словам, по началу слова, `difflib` — что сочтёте разумным. Если подходящих вершин несколько или нет ни одной — `None`. Слияние не должно зависеть от порядка документов.
* **2.2 `add_extraction(doc_id, extraction)` (0.25).** Добавляет в граф `self.graph` сущности и связи одного документа — то, что для него вернула `parse_extraction`: каждую сущность — через `resolve` (новая вершина или существующая), каждую связь — ребром между вершинами её концов. Повторное добавление того же документа не создаёт дублей.
* **2.3 `find(query, etype=None, limit=5) -> list[node]` (0.25).** Ищет в графе вершины, которые упомянуты в запросе, и возвращает их идентификаторы: запрос может быть названием («гастротур») или вопросом целиком. Самые подходящие — первыми.
* **2.4 `expand(nodes, depth=2, relations=None, max_edges=30) -> list[dict]` (1).** Собирает подграф — часть графа вокруг заданных вершин, по которой можно пройти цепочку. В него входят рёбра на расстоянии до `depth` шагов, в обе стороны. `relations` ограничивает, по каким связям идти. Не больше `max_edges` рёбер, без повторов. Ребро — словарь `source`, `relation`, `target`, `doc_id`, `evidence`.
* **2.5 `evidence(edges) -> str` (0.25).** Превращает список рёбер из `expand` в текст, который инструмент поиска вернёт модели в части 3: по строке на ребро — имена и типы концов, связь, `[doc_id]` и цитата. Для пустого списка — понятная фраза.

In [48]:
class KnowledgeGraph:
    def __init__(self):
        self.graph = nx.MultiDiGraph()
        self.extractions = {}

    def resolve(self, name: str, etype: str) -> Optional[str]:
        # TODO 2.1
        name = self._normalize(name)
        exact, similar = [], []
        for node, data in self.graph.nodes(data=True):
            if data["type"] != etype:
                continue
            if name in data["aliases"]:
                exact.append(node)
            elif any((etype == "Гид" and (name in alias.split() or alias in name.split())) or (etype == "Отель" and {name, alias} == {"петроградка", "петроградская сторона"}) for alias in data["aliases"]):
                similar.append(node)
        matches = exact or similar
        return matches[0] if len(matches) == 1 else None
    @staticmethod
    def _normalize(name: str) -> str:
        return re.sub(r"^(отель|гостиница|экскурсия|тур|гид) ", "", " ".join(tokenize(name)).replace("ё", "е"))

    def add_extraction(self, doc_id: str, extraction: dict) -> None:
        # TODO 2.2
        self.extractions[doc_id] = extraction
        self.graph.clear()
        mentions = defaultdict(set)
        for source_id, item in self.extractions.items():
            for entity in item["entities"]:
                mentions[entity["type"], entity["name"]].add(source_id)

        def sort_key(mention):
            etype, name = mention
            alias = self._normalize(name)
            return etype, -len(alias.split()), -len(alias), alias, name

        nodes = {}
        for etype, name in sorted(mentions, key=sort_key):
            node = self.resolve(name, etype)
            if node is None:
                node = str(self.graph.number_of_nodes())
                self.graph.add_node(node, name=name, type=etype, aliases=set(), doc_ids=set())
            data = self.graph.nodes[node]
            data["name"] = max((data["name"], name), key=lambda s: (len(s), s))
            data["aliases"].add(self._normalize(name))
            data["doc_ids"].update(mentions[etype, name])
            nodes[etype, name] = node

        for source_id, item in sorted(self.extractions.items()):
            by_name = {e["name"]: nodes[e["type"], e["name"]] for e in item["entities"]}
            for edge in item["relations"]:
                relation = edge["relation"]
                self.graph.add_edge(
                    by_name[edge["source"]], by_name[edge["target"]],
                    key=(relation, source_id), relation=relation,
                    doc_id=source_id, evidence=edge["evidence"]
                )

    def find(self, query: str, etype: Optional[str] = None, limit: int = 5) -> list[str]:
        # TODO 2.3
        query = self._normalize(query)
        scores = {}
        for node, data in self.graph.nodes(data=True):
            if etype is None or data["type"] == etype:
                for alias in data["aliases"]:
                    if f" {alias} " in f" {query} " or f" {alias} ".startswith(f" {query} "):
                        scores[node] = max(scores.get(node, (False, 0)), (query == alias, min(len(query), len(alias))))
        return sorted(scores, key=scores.get, reverse=True)[:limit]

    def expand(self, nodes: list[str], depth: int = 2, relations: Optional[list[str]] = None, max_edges: int = 30) -> list[dict]:
        # TODO 2.4
        frontier = list(dict.fromkeys(nodes))
        visited, edges = set(frontier), {}
        for _ in range(depth):
            following = []
            for node in frontier:
                for source, target, key, edge in (*self.graph.out_edges(node, keys=True, data=True), *self.graph.in_edges(node, keys=True, data=True)):
                    if relations is not None and edge["relation"] not in relations:
                        continue
                    if len(edges) == max_edges:
                        return list(edges.values())
                    edges[source, target, key] = {"source": source, "target": target, **edge}
                    neighbor = target if node == source else source
                    if neighbor not in visited:
                        visited.add(neighbor)
                        following.append(neighbor)
            frontier = following
        return list(edges.values())

    def evidence(self, edges: list[dict]) -> str:
        # TODO 2.5
        return "\n".join(f"{self.graph.nodes[e['source']]['name']} ({self.graph.nodes[e['source']]['type']}) —{e['relation']}→ {self.graph.nodes[e['target']]['name']} ({self.graph.nodes[e['target']]['type']}) [{e['doc_id']}] «{e['evidence']}»" for e in edges) or "В графе нет подходящих связей."

In [49]:
for task in ["2.1", "2.2", "2.3", "2.4", "2.5"]:
    check(task, KnowledgeGraph=KnowledgeGraph)

✅ Задание 2.1: базовые тесты пройдены
✅ Задание 2.2: базовые тесты пройдены
✅ Задание 2.3: базовые тесты пройдены
✅ Задание 2.4: базовые тесты пройдены
✅ Задание 2.5: базовые тесты пройдены


Соберём граф из своих извлечений и посмотрим на него. Список «похожих имён» — кандидаты на ошибки слияния: одна сущность разбита на две вершины или, наоборот, разные сущности склеены.

In [50]:
from difflib import SequenceMatcher

kg = KnowledgeGraph()
for item in EXTRACTIONS:
    if item["parsed"] is not None:
        kg.add_extraction(item["doc_id"], item["parsed"])

nodes = kg.graph.nodes(data=True)
print("Вершин:", kg.graph.number_of_nodes(), "| рёбер:", kg.graph.number_of_edges())
print("Вершины по типам:", dict(Counter(d["type"] for _, d in nodes)))
print("Рёбра по связям:", dict(Counter(d["relation"] for _, _, d in kg.graph.edges(data=True))))
lonely = [d["name"] for n, d in nodes if kg.graph.degree(n) == 0]
print("Вершины без связей:", lonely or "нет")

def _plain(name: str) -> str:
    name = re.sub(r"[«»'\"]", "", name.lower().replace("ё", "е"))
    return re.sub(r"^(отель|гостиница|экскурсия|тур|гид)\s+", "", name).strip()


print("\nПохожие имена одного типа (возможно, одна сущность разбита на две):")
by_type = defaultdict(list)
for n, d in nodes:
    by_type[d["type"]].append(d["name"])
similar = 0
for etype, names in by_type.items():
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            pa, pb = _plain(a), _plain(b)
            if pa in pb or pb in pa or len(os.path.commonprefix([pa, pb])) >= 5 or SequenceMatcher(None, pa, pb).ratio() > 0.8:
                print(f"  {etype}: «{a}» / «{b}»")
                similar += 1
if not similar:
    print("  не найдено")

print("\nСклеенные вершины (несколько разных имён):")
for n, d in nodes:
    if len(d["aliases"]) > 1:
        print(f"  {d['type']}: {d['name']} ← {sorted(d['aliases'])}")

guide = kg.find("Алия Гарипова", etype="Гид")
if guide:
    print("\nПример подграфа:")
    print(kg.evidence(kg.expand(guide, depth=1)))

Вершин: 56 | рёбер: 108
Вершины по типам: {'Гид': 11, 'Город': 6, 'Отель': 12, 'Тур': 5, 'Экскурсия': 15, 'Язык': 7}
Рёбра по связям: {'ведёт': 15, 'работает_в': 11, 'говорит_на': 25, 'находится_в': 27, 'проживание_в': 7, 'включает': 11, 'стартует_от': 12}
Вершины без связей: нет

Похожие имена одного типа (возможно, одна сущность разбита на две):
  не найдено

Склеенные вершины (несколько разных имён):
  Гид: Георгий Кикнадзе ← ['георгий кикнадзе', 'кикнадзе']
  Гид: Елена Соколова ← ['елена соколова', 'соколова']
  Гид: Олег Строганов ← ['олег строганов', 'строганов']
  Гид: Алия Гарипова ← ['алия', 'алия гарипова']
  Гид: Дмитрий Орлов ← ['дмитрий', 'дмитрий орлов']
  Гид: Ирина Волкова ← ['волкова', 'ирина волкова']
  Гид: Ашот Мкртчян ← ['ашот', 'ашот мкртчян']
  Гид: Лилит Акопян ← ['акопян', 'лилит акопян']
  Гид: Мерве Йылдыз ← ['мерве', 'мерве йылдыз']
  Гид: Нино Беридзе ← ['нино', 'нино беридзе']
  Гид: Марат Сафин ← ['марат сафин', 'сафин']
  Отель: Петроградская сторона ← 

## 3. Агент с двумя видами поиска (2.5 балла)

Поиск по тексту дан готовым: BM25 по документам, как на семинаре. Он возвращает три самых похожих документа целиком.

In [51]:
BM25 = BM25Okapi([tokenize(doc["title"] + " " + doc["text"]) for doc in DOCS])


def text_search(query: str, k: int = 3) -> str:
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    hits = [DOCS[i] for i in top if scores[i] > 0]
    if not hits:
        return "По тексту ничего не найдено."
    return "\n\n".join(f"[{doc['doc_id']}] {doc['title']}\n{doc['text']}" for doc in hits)


text_tool = StructuredTool.from_function(
    func=text_search, name="text_search", handle_tool_error=True,
    description="Поиск по документам турагентства: карточки отелей, гидов, экскурсий, туров и правила агентства. "
                "Возвращает три самых похожих документа с [doc_id].",
)
print(text_search("языки экскурсии Гастротур по Казани"))

[e_kzn_food] Экскурсия «Гастротур по Казани»
«Гастротур по Казани» — авторская экскурсия в Казани: чак-чак, эчпочмак и губадия. Длительность 3 ч, стоимость 2900 рублей с человека.

Встреча у входа на Колхозный рынок. Ведёт экскурсию Алия.

[t_volga] Тур «Казанские выходные»
«Казанские выходные» — тур на 3 дня, маршрут: Казань. Стоимость 32000 рублей на человека.

Проживание: «Кремлёвский двор». В программу входят экскурсии «Гастротур по Казани»; «Кремль и Кул-Шариф». Остальное время свободное.

[g_safin] Гид Марат Сафин
Марат Сафин работает с нами в Казани уже 6 лет. Языки: русский, немецкий.

Отзывы туристов о гиде можно прочитать в личном кабинете.


### 3.1. Инструмент поиска по графу (0.75 балла)

Напишите `make_graph_tool(kg) -> BaseTool` — функцию, которая возвращает инструмент LangChain с именем `graph_search`.

Способ — любой из ДЗ 1: `StructuredTool.from_function` со схемой аргументов на Pydantic, `@tool` или класс-наследник `BaseTool`.

Требования:

* аргументы:
  * `entities` — список названий сущностей, как они звучат в вопросе (обязательный);
  * `depth` — глубина от 1 до 3; можно добавить и другие, например фильтр связей;
* инструмент находит вершины через `kg.find`, раскрывает подграф через `kg.expand` и возвращает `kg.evidence`; про сущности, которых в графе нет, прямо говорит модели;
* описание объясняет модели, какие вопросы решаются графом, а какие — поиском по тексту;
* неверные аргументы не роняют агента: модель получает текст ошибки.

In [52]:
def make_graph_tool(kg: KnowledgeGraph) -> BaseTool:
    # TODO 3.1
    class GraphSearchArgs(BaseModel):
        entities: list[str]
        depth: int = Field(default=2, ge=1, le=3)
    def graph_search(entities: list[str], depth: int = 2) -> str:
        nodes, missing = [], []
        for entity in entities:
            found = kg.find(entity, limit=1)
            if found:
                nodes.extend(found)
            else:
                missing.append(entity)
        return kg.evidence(kg.expand(nodes, depth=depth)) + ("\nНе найдено в графе: " + ", ".join(missing) if missing else "")
    return StructuredTool.from_function(graph_search, args_schema=GraphSearchArgs, handle_validation_error="Передай entities списком названий и depth целым числом от 1 до 3.", description="Ищи связи между турами, отелями, экскурсиями, гидами и языками. Передай названия сущностей из вопроса и глубину от 1 до 3. Цены, расписание и правила ищи инструментом text_search.")


check("3.1", make_graph_tool=make_graph_tool, KnowledgeGraph=KnowledgeGraph)
graph_tool = make_graph_tool(kg)
print(graph_tool.invoke({"entities": ["Гастротур по Казани"], "depth": 2}))

✅ Задание 3.1: базовые тесты пройдены
Гастротур по Казани (Экскурсия) —находится_в→ Казань (Город) [e_kzn_food] ««Гастротур по Казани» — авторская экскурсия в Казани: чак-чак, эчпочмак и губадия.»
Алия Гарипова (Гид) —ведёт→ Гастротур по Казани (Экскурсия) [e_kzn_food] «Ведёт экскурсию Алия.»
Казанские выходные (Тур) —включает→ Гастротур по Казани (Экскурсия) [t_volga] «В программу входят экскурсии «Гастротур по Казани»; «Кремль и Кул-Шариф».»
Кремль и Кул-Шариф (Экскурсия) —находится_в→ Казань (Город) [e_kzn_kremlin] ««Кремль и Кул-Шариф» — авторская экскурсия в Казани: кремль, башня Сююмбике и мечеть Кул-Шариф.»
Старо-Татарская слобода (Экскурсия) —находится_в→ Казань (Город) [e_kzn_sloboda] ««Старо-Татарская слобода» — авторская экскурсия в Казани»
Алия Гарипова (Гид) —работает_в→ Казань (Город) [g_garipova] «Алия Гарипова — гид «Попутного ветра» в Казани, стаж 10 лет.»
Марат Сафин (Гид) —работает_в→ Казань (Город) [g_safin] «Марат Сафин работает с нами в Казани уже 6 лет.»
Кабан (О

### 3.2. Промпт и агент (1.25 балла)

1. `AGENT_PROMPT`: когда искать по тексту, когда по графу, когда сочетать; факты — только из результатов инструментов; после каждого факта — `[doc_id]`; если ничего не нашлось — сказать, что в базе этого нет.
2. `build_agent(llm, tools)` — агент на `create_agent` с этим промптом.
3. `ask(agent, question, max_steps=8) -> dict` — вопрос с пустой историей, не больше `max_steps` обращений к модели. Возвращает `answer`, `messages`, `tool_calls` (список пар (имя, аргументы) по порядку), `cited_doc_ids` (`extract_citations`), `llm_calls` (число ответов модели). При превышении лимита — понятный ответ без исключения.

In [53]:
AGENT_PROMPT = """
Ты отвечаешь на вопросы о турагентстве.
Используй только доступные инструменты и опирайся только на найденные в них факты.
Для цен, расписания, правил и сведений из одной карточки используй text_search.
Если нужно связать тур, экскурсию, отель, гида или язык, используй graph_search, начиная с названий из вопроса.
Если для ответа не хватает деталей, найди через text_search карточки сущностей из графа.
Если граф недоступен или не дал результата, попробуй text_search по отдельным сущностям и уточни запрос по найденным карточкам.
Проверяй, что цепочка связей относится именно к сущностям из вопроса.
После каждого факта ставь [doc_id] источника и приводи только реальные идентификаторы из результатов поиска.
Если данных для ответа нет, скажи, что этой информации нет в базе.
"""


def build_agent(llm, tools: list) -> Any:
    # TODO 3.2
    return create_agent(llm, tools=tools, system_prompt=AGENT_PROMPT)


def ask(agent, question: str, max_steps: int = 8) -> dict:
    # TODO 3.2
    messages = [HumanMessage(content=question)]
    responses = []
    for state in agent.stream({"messages": messages}, config={"recursion_limit": 2 * max_steps + 1}, stream_mode="values"):
        messages = state["messages"]
        if isinstance(messages[-1], AIMessage):
            responses.append(messages[-1])
            if len(responses) == max_steps:
                break
    answer = (responses[-1].text if not responses[-1].tool_calls else "") or "Не удалось ответить за отведённое число шагов."
    return {"answer": answer, "messages": messages, "tool_calls": [(call["name"], call["args"]) for message in responses for call in message.tool_calls], "cited_doc_ids": extract_citations(answer), "llm_calls": len(responses)}


check("3.2", build_agent=build_agent, ask=ask, AGENT_PROMPT=AGENT_PROMPT)

✅ Задание 3.2: базовые тесты пройдены


In [45]:
graph_agent = build_agent(llm, [text_tool, graph_tool])
for qid in ["q03", "q07", "q15", "q21"]:
    item = next(i for i in EVAL if i["qid"] == qid)
    result = ask(graph_agent, item["question"])
    print(f"=== {qid} ({item['hops']} шаг.): {item['question']}")
    for message in result["messages"]:
        if isinstance(message, AIMessage):
            for call in message.tool_calls:
                print(f"  Вызов: {call['name']}({call['args']})")
        elif isinstance(message, ToolMessage):
            print(f"    Результат: {message.text[:200].replace(chr(10), ' / ')}")
    print(f"  Ответ: {result['answer']}\n")

=== q03 (1 шаг.): Сколько стоит ночь в отеле «Кабан» в Казани?
  Вызов: text_search({'query': 'Казань отель «Кабан» цена за ночь', 'k': 3})
    Результат: [h_kaban] Гостиница «Кабан» / В Казани мы часто селим туристов в «Кабан»: на берегу озера Кабан, рядом Старо-Татарская слобода. Цена — от 5100 рублей за ночь. /  / Заезд с 14:00, выезд до 12:00. Бронировани
  Ответ: Ночь в гостинице «Кабан» в Казани стоит от 5 100 рублей. [h_kaban]

=== q07 (2 шаг.): На каких языках можно провести экскурсию «Гастротур по Казани»?
  Вызов: graph_search({'entities': ['Гастротур по Казани'], 'depth': 2})
    Результат: Гастротур по Казани (Экскурсия) —находится_в→ Казань (Город) [e_kzn_food] ««Гастротур по Казани» — авторская экскурсия в Казани: чак-чак, эчпочмак и губадия.» / Алия Гарипова (Гид) —ведёт→ Гастротур по 
  Ответ: «Гастротур по Казани» можно провести на русском, татарском или английском языке [e_kzn_food] [g_garipova].

=== q15 (3 шаг.): На каких языках говорят гиды экскурсий, входящих в ту

### 3.3. Что делает агент (0.5 балла)

По выводу выше (3–5 предложений): какой поиск агент выбрал для вопроса, на который отвечает один документ, и для вопросов, где ответ складывается из нескольких связей (экскурсия → гид → языки); хватило ли ему графа или он дочитывал документы; что он ответил на вопрос, ответа на который в базе нет.

**Ваш ответ:**
Для ответа на вопрос q03 агент использовал только поиск по тексту и корректно нашел нужный документ.

Для ответа на остальные вопросы агент сначала воспользовался поиском по графу, чего ему хватило в q07 и q15.

Для ответа на вопрос, информации по которому нет в базе данных, агент использовал поиск как по графу, так и по тексту чтобы убедиться, что этой информации действительно нигде нет. Он корректно уточнил, что этой информации в базе данных нет и предложил альтернативу.

Обычный RAG для сравнения дан готовым: один текстовый поиск по вопросу и один запрос к модели.

In [55]:
CLASSIC_PROMPT = (
    "Ответь на вопрос только по фрагментам базы турагентства. После каждого факта ставь [doc_id]. "
    "Если ответа во фрагментах нет — так и скажи.\n\nФрагменты:\n{context}\n\nВопрос: {question}"
)


def classic_rag(question: str) -> dict:
    context = text_search(question)
    reply = llm.invoke(CLASSIC_PROMPT.format(context=context, question=question)).text
    return {"answer": reply, "messages": [], "tool_calls": [("text_search", {"query": question})],
            "cited_doc_ids": extract_citations(reply), "llm_calls": 1}

## 4. Сравнение (1.25 балла)

Сравним три системы на 10 вопросах из 24 (`COMPARE_QIDS`, по несколько на каждое число шагов), чтобы не тратить много запросов: обычный RAG, агента только с текстовым поиском и агента с текстом и графом. Метрики по вопросу считает готовая `score_answer`:

* `facts_recall` — доля ключевых фактов, найденных в ответе;
* `citation_ok` — процитирован ли хотя бы один нужный документ (для вопросов без ответа в базе — `None`);
* `honest` — для вопросов без ответа в базе: признал ли агент, что информации нет, и не сослался ли на источники; для остальных — `None`;
* `graph_calls`, `text_calls`, `llm_calls` — сколько было вызовов графа, текста и модели.

In [56]:
METRICS = ["facts_recall", "citation_ok", "honest", "graph_calls", "text_calls", "llm_calls"]
HOP_NAMES = {0: "нет в базе", 1: "1 шаг", 2: "2 шага", 3: "3 шага"}


def score_answer(item: dict, result: dict) -> dict:
    answer = (result["answer"] or "").lower().replace("ё", "е")
    facts = item["key_facts"]
    recall = (sum(any(v.lower().replace("ё", "е") in answer for v in variants) for variants in facts) / len(facts)) if facts else None
    cited = set(result["cited_doc_ids"])
    return {
        "facts_recall": recall,
        "citation_ok": bool(cited & set(item["gold_doc_ids"])) if item["gold_doc_ids"] else None,
        "honest": (has_no_info(result["answer"]) and not cited) if item["hops"] == 0 else None,
        "graph_calls": sum(name == "graph_search" for name, _ in result["tool_calls"]),
        "text_calls": sum(name == "text_search" for name, _ in result["tool_calls"]),
        "llm_calls": result["llm_calls"],
    }


def run_eval(system_fn: Callable, items: list[dict]) -> list[dict]:
    results = parallel_map(lambda item: system_fn(item["question"]), items, "вопросы")
    rows = []
    for item, result in zip(items, results):
        rows.append({"qid": item["qid"], "hops": item["hops"], **score_answer(item, result),
                     "answer": result["answer"], "tool_calls": result["tool_calls"]})
    return rows


def print_rows(title: str, rows: list[dict]) -> None:
    print(f"\n{title}")
    mark = lambda v: "—" if v is None else ("да" if v is True else ("нет" if v is False else f"{v:.2f}"))
    for row in rows:
        print(f"{row['qid']} {HOP_NAMES[row['hops']]:<10} факты {mark(row['facts_recall']):<5} цитата {mark(row['citation_ok']):<4} "
              f"честно {mark(row['honest']):<4} граф {row['graph_calls']} текст {row['text_calls']} | {row['answer'][:70]!r}")

Сводку по группам вопросов считает функция `summarize`: для каждой группы по числу шагов и для всех вопросов вместе — число вопросов и среднее по каждой метрике (`None` в среднем не участвует).

In [57]:
def summarize(rows: list[dict]) -> dict:
    groups: dict[str, list[dict]] = defaultdict(list)
    for row in rows:
        groups[HOP_NAMES[row["hops"]]].append(row)
        groups["ВСЕГО"].append(row)
    table = {}
    for name, items in groups.items():
        values = {"n": len(items)}
        for metric in METRICS:
            present = [float(item[metric]) for item in items if item[metric] is not None]
            values[metric] = round(sum(present) / len(present), 3) if present else None
        table[name] = values
    return table

In [58]:
COMPARE_QIDS = ["q03", "q05", "q07", "q10", "q13", "q15", "q16", "q17", "q19", "q21"]
COMPARE = [item for item in EVAL if item["qid"] in COMPARE_QIDS]

text_agent = build_agent(llm, [text_tool])
classic_rows = run_eval(classic_rag, COMPARE)
text_rows = run_eval(lambda q: ask(text_agent, q), COMPARE)
graph_rows = run_eval(lambda q: ask(graph_agent, q), COMPARE)

print_rows("Обычный RAG", classic_rows)
print_rows("Агент: только текст", text_rows)
print_rows("Агент: текст + граф", graph_rows)

for title, rows in [("Обычный RAG", classic_rows), ("Агент: только текст", text_rows), ("Агент: текст + граф", graph_rows)]:
    table = summarize(rows)
    print(f"\n{title}")
    print(f"{'группа':<12}{'n':>3}" + "".join(f"{m:>14}" for m in METRICS))
    for name in sorted(table, key=lambda n: (n == "ВСЕГО", n)):
        values = table[name]
        print(f"{name:<12}{values['n']:>3}" + "".join(f"{'—' if values[m] is None else round(values[m], 2):>14}" for m in METRICS))

вопросы: готово 2/10
вопросы: готово 4/10
вопросы: готово 6/10
вопросы: готово 8/10
вопросы: готово 10/10
вопросы: готово 2/10
вопросы: готово 4/10
вопросы: готово 6/10
вопросы: готово 8/10
вопросы: готово 10/10
вопросы: готово 2/10
вопросы: готово 4/10
вопросы: готово 6/10
вопросы: готово 8/10
вопросы: готово 10/10

Обычный RAG
q03 1 шаг      факты 1.00  цитата да   честно —    граф 0 текст 1 | 'Ночь в отеле «Кабан» в Казани стоит от 5100 рублей [h_kaban].'
q05 1 шаг      факты 1.00  цитата да   честно —    граф 0 текст 1 | 'При отмене тура за 10 дней до выезда вернут 50% его стоимости [policy_'
q07 2 шага     факты 1.00  цитата да   честно —    граф 0 текст 1 | '«Гастротур по Казани» ведёт Алия. [e_kzn_food] Если речь об Алие Гарип'
q10 2 шага     факты 1.00  цитата да   честно —    граф 0 текст 1 | 'Экскурсию «Кремль и Кул-Шариф» ведёт Сафин [e_kzn_kremlin]. Говорит ли'
q13 2 шага     факты 0.00  цитата да   честно —    граф 0 текст 1 | 'Участники тура «Казанские выходные» живут в о

### 4.1. Выводы (1.25 балла)

По сводкам и построчным выводам, ссылаясь на цифры и номера вопросов:

1. На каких вопросах граф помог и почему: что агент с графом сделал иначе, чем агент только с текстом?
2. Где граф не помог или помешал? Есть ли вопросы, где агент с текстом справился лучше? Чем это объясняется — ошибками вашего графа, выбором инструмента или чем-то ещё?
3. Сколько стоит каждая система (вызовы инструментов и модели) и что бы вы выбрали для такой базы?

**Ваш ответ:**

1. На q07, q10 и q13 оба агента получили полноту фактов 1.00. Текстовый агент каждый раз выполнил два текстовых поиска, а агенту с графом хватило одного обращения к графу. Например, в q07 нужно связать экскурсию с гидом и узнать, на каких языках он говорит. В графе обе связи можно получить за один вызов. На q15 разница еще существеннее: текстовый агент выполнил семь поисков, а агент с графом - один, при одинаковой полноте 1.00. У обычного RAG с одним текстовым поиском полнота на q15 составила 0.75.

2. На q19 текстовый агент после четырех поисков получил 1.00, а агент с графом после одного обращения к графу - 0.00. Вывод обрезан, поэтому надо смотреть подробнее, в чем конкретно причина провала агента с графом - в следующем пункте. На q03 оба агента использовали только текстовый поиск и правильно указали цену, но записали её как "5 100" и получили 0.00. Обычный RAG написал "5100" и получил 1.00.

3. В среднем на вопрос обычный RAG выполнял один текстовый поиск и один вызов модели. У текстового агента было 3.0 текстовых поиска и 3.1 вызова модели, у агента с графом - 0.8 обращения к графу, 0.4 текстового поиска и 2.2 вызова модели. Средняя полнота фактов составила 0.66, 0.89 и 0.78 соответственно. Для этой базы я бы выбрала агента с текстом и графом: он может получить цену или правило из карточки и за один поиск пройти по связям между туром, экскурсией и гидом. В этом прогоне он уступил текстовому агенту по полноте, но потребовал в среднем 1.2 вызова инструментов на вопрос против 3.0. Так как вызовы модели дороже поиска, надо выбрать систему, в которой модель вызывается реже, но при этом сохраняется приличный recall.

## 5. Разбор ошибок (1.75 балла)

### 5.1. Три ошибки по уровням (1.75 балла)

Выберите из прогона агента с графом три вопроса, где что-то пошло не так. Для каждого: что произошло (по вызовам, найденному и ответу) и на каком уровне ошибка — извлечение, слияние сущностей, поиск вершин, раскрытие подграфа, выбор инструмента агентом или генерация ответа, — и как бы вы её исправили: что поменять и на каком уровне (одно-два предложения, код писать не нужно). Ячейка ниже печатает подробности — впишите номера вопросов.

Если явных ошибок меньше трёх, доберите вопросы, где агент ответил верно, но неоптимально: лишние или повторные вызовы, не тот инструмент с первой попытки, подграф из сотни рёбер ради одного факта, нет ссылки на нужный документ. Если и таких нет — сломайте один уровень сами: например, пусть `resolve` всегда возвращает `None` (сущности не сливаются) или `expand` идёт только на один шаг, — пересоберите граф и разберите, какие вопросы сломались и почему.

In [59]:
ERROR_QIDS = ["q03", "q19", "q21"]  # TODO 5.1: три номера вопросов, например ["q16", "q17", "q19"]

In [60]:
for qid in ERROR_QIDS:
    row = next(r for r in graph_rows if r["qid"] == qid)
    item = next(i for i in EVAL if i["qid"] == qid)
    print(f"===== {qid}: {item['question']}")
    print("Нужные документы:", item["gold_doc_ids"], "| ключевые факты:", item["key_facts"])
    print("Вызовы:", row["tool_calls"])
    for name, args in row["tool_calls"]:
        if name == "graph_search":
            print("  Граф нашёл:", [kg.graph.nodes[n]["name"] for e in args.get("entities", []) for n in kg.find(e, limit=1)])
    print("Ответ:", row["answer"], "\n")

===== q03: Сколько стоит ночь в отеле «Кабан» в Казани?
Нужные документы: ['h_kaban'] | ключевые факты: [['5100']]
Вызовы: [('text_search', {'query': 'отель «Кабан» Казань стоимость за ночь цена', 'k': 3})]
Ответ: Ночь в гостинице «Кабан» в Казани стоит от 5 100 рублей [h_kaban]. 

===== q19: На каком языке, кроме русского, говорят гиды всех экскурсий тура «Казанские выходные»?
Нужные документы: ['t_volga', 'e_kzn_food', 'e_kzn_kremlin', 'g_garipova', 'g_safin'] | ключевые факты: [['ни на каком', 'ни на одном', 'только на русском', 'только русский', 'общего языка нет', 'нет общего', 'нет такого языка']]
Вызовы: [('graph_search', {'entities': ['Казанские выходные'], 'depth': 3})]
  Граф нашёл: ['Казанские выходные']
Ответ: Кроме русского, **общего языка у гидов нет** [g_garipova] [g_safin]. В тур «Казанские выходные» входят две экскурсии [t_volga]:

- «Гастротур по Казани» ведёт Алия Гарипова [e_kzn_food]. Кроме русского, она говорит на татарском и английском [g_garipova].
- «Кремль и К

Для каждого из трёх вопросов: что произошло, на каком уровне ошибка и как бы вы её исправили.

**Ваш ответ:**

1. q03: За один вызов text_search агент нашел карточку [h_kaban] и верно указал цену "от 5 100 рублей", но получил facts_recall = 0.00: из-за узкого неразрывного пробела строка 5100 не встретилась в ответе. Я бы исправила метрику, убирая разделители тысяч между цифрами перед сравнением ответа с ключевым фактом.

2. q19: За один вызов graph_search с глубиной 3 агент нашел тур, в ответе сослался на все пять нужных документов и правильно объяснил, что Гарипова говорит на татарском и английском, а Сафин — на немецком, поэтому общего иностранного языка у них нет. Оценка 0.00 получилась из-за разницы в формулировках: в ответе "общего языка у гидов нет", а в эталоне — "общего языка нет". Я бы добавила "общего языка у гидов нет" в список допустимых вариантов для q19.

3. q21: Поиск по графу не нашел вершину "японский язык". После двух текстовых поисков агент верно указал, что наличие японоязычного гида не подтверждено, но сослался на карточки трех других гидов и правило заказа экскурсии. Эти источники не подтверждают наличие такого гида, а любая ссылка уже даёт honest = 0. Я бы уточнила промпт: если прямого подтверждения нет, отвечать без ссылок одной фразой - "В базе нет информации об этом".